# 01. 복원 구조와 작은 오토인코더

In [ ]:
import torch  # 파이토치
import torch.nn as nn  # 신경망 층 모음

encoder = nn.Linear(2, 1, bias=False)  # 인코더: 2칸 → 1칸, 편향 없음
decoder = nn.Linear(1, 2, bias=False)  # 디코더: 1칸 → 2칸, 편향 없음
encoder.weight = nn.Parameter(torch.tensor([[0.2, 0.6]]))  # 인코더 가중치(e) 직접 지정, 모양 (1, 2)
decoder.weight = nn.Parameter(torch.tensor([[1.5], [0.5]]))  # 디코더 가중치(d) 직접 지정, 모양 (2, 1)
x = torch.tensor([[3.0, 1.0]])  # 입력(x), 모양 (1, 2)

with torch.no_grad():  # 기울기 기록 없이 계산
    z = encoder(x)  # 잠재 표현(z)
    rec = decoder(z)  # 복원값(x̂)
    mse = nn.MSELoss()(rec, x)  # 평균제곱오차(MSE)
print(z, rec, mse)  # 결과 출력

tensor([[1.2000]]) tensor([[1.8000, 0.6000]]) tensor(0.8000)


# 02. 역전파와 기울기 계산 (없음)

# 03. 경사 하강법과 파라미터 갱신

In [1]:
import torch  # 파이토치
import torch.nn as nn  # 신경망 층 모음

encoder = nn.Linear(2, 1, bias=False)  # 인코더: 2칸 → 1칸
decoder = nn.Linear(1, 2, bias=False)  # 디코더: 1칸 → 2칸
encoder.weight = nn.Parameter(torch.tensor([[0.5, 0.2]]))  # 인코더 가중치(e) 지정
decoder.weight = nn.Parameter(torch.tensor([[1.0], [0.5]]))  # 디코더 가중치(d) 지정
model = nn.Sequential(encoder, decoder)  # 인코더 → 디코더 연결
x = torch.tensor([[2.0, 3.0]])  # 입력(x)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # 경사 하강법, 학습률(η)=0.1

loss = nn.MSELoss()(model(x), x)  # 순전파 후 평균제곱오차(L)
loss.backward()  # 역전파: 모든 기울기 계산
print(encoder.weight.grad, decoder.weight.grad)  # 인코더·디코더 가중치 기울기
optimizer.step()  # 가중치 갱신
print(encoder.weight.data, decoder.weight.data)  # 갱신된 가중치

tensor([[-3.0000, -4.5000]]) tensor([[-0.6400],
        [-3.5200]])
tensor([[0.8000, 0.6500]]) tensor([[1.0640],
        [0.8520]])


# 04. 병목과 정보 손실


In [4]:
import numpy as np

x = np.array([2.0, 6.0, 5.0, 1.0])  # 입력(x) 네 칸
z = np.array([(x[0] + x[1]) / 2, (x[2] + x[3]) / 2])  # 잠재 표현(z): 앞 두 칸 평균, 뒤 두 칸 평균
rec = np.array([z[0], z[0], z[1], z[1]])  # 복원값(x̂): z의 각 칸을 두 번씩 복사
mse = ((rec - x) ** 2).mean()  # 평균제곱오차: 차이 제곱의 평균
print(z, rec, mse)  # 결과 출력

[4. 3.] [4. 4. 3. 3.] 4.0
